# Análise de Negociações do Trocabook

Este experimento tem como objetivo analisar o comportamento histórico das negociações do Trocabook, utilizando dados simulados para validar indicadores, técnicas estatísticas e regras de geração de insights antes da integração com os dados reais da plataforma.

## 1. Bibliotecas utilizadas

O experimento utiliza a biblioteca Pandas para manipulação, transformação e análise dos dados de anúncios e negociações.

A biblioteca permite organizar os dados em DataFrames, realizar agrupamentos, relacionar informações provenientes de diferentes coleções e calcular indicadores relacionados ao comportamento das negociações.

A biblioteca NumPy é utilizada para manipulação dos conjuntos numéricos empregados nos cálculos estatísticos, permitindo trabalhar de forma eficiente com os valores de negociações finalizadas e não finalizadas.

A biblioteca SciPy é utilizada na etapa de análise estatística dos indicadores. A função `betaln` permite calcular o logaritmo da função Beta de forma numericamente estável, enquanto a função `minimize` é utilizada para estimar, por máxima verossimilhança, os parâmetros da distribuição Beta-Binomial.

Neste primeiro experimento não são utilizados modelos de aprendizado de máquina. O objetivo é realizar análises descritivas e diagnósticas dos dados, complementadas por técnicas estatísticas para avaliar a confiabilidade dos indicadores observados.

In [1]:
import pandas as pd
import numpy as np

from scipy.optimize import minimize
from scipy.special import betaln

## 2. Dados utilizados no experimento

Para validar inicialmente a análise das negociações, foram criados dados
simulados de anúncios e negociações com estrutura semelhante aos documentos
armazenados no Firestore pelo Trocabook.

Os anúncios fornecem informações sobre os livros envolvidos nas negociações,
principalmente suas categorias e o tipo de negociação permitido.

As negociações registram informações como o anúncio negociado, o tipo da
negociação, seu status e as datas de início e finalização.

Os dados simulados incluem diferentes situações de TROCA e VENDA, permitindo
comparar a quantidade de negociações iniciadas, finalizadas, recusadas e
canceladas.

Esses dados permitem testar as técnicas de análise antes da integração do
microsserviço com os dados reais do Trocabook.

### Anúncios simulados

Os anúncios representam os livros disponibilizados pelos usuários no
Trocabook.

Para este experimento são utilizados apenas os atributos necessários para
relacionar os anúncios às negociações e realizar análises por categoria.

O campo `categorias` possui uma lista porque um mesmo livro pode pertencer
a mais de uma categoria.

In [2]:
anuncios = pd.DataFrame([
    {
        "id": "A1",
        "uidLivro": "L1",
        "tipoNegociacao": "AMBOS",
        "categorias": ["Fantasia", "Aventura"],
        "status": "FINALIZADO"
    },
    {
        "id": "A2",
        "uidLivro": "L2",
        "tipoNegociacao": "VENDA",
        "categorias": ["Tecnologia", "Programação"],
        "status": "FINALIZADO"
    },
    {
        "id": "A3",
        "uidLivro": "L3",
        "tipoNegociacao": "TROCA",
        "categorias": ["Romance"],
        "status": "ATIVO"
    },
    {
        "id": "A4",
        "uidLivro": "L4",
        "tipoNegociacao": "AMBOS",
        "categorias": ["Fantasia"],
        "status": "FINALIZADO"
    },
    {
        "id": "A5",
        "uidLivro": "L5",
        "tipoNegociacao": "VENDA",
        "categorias": ["História"],
        "status": "ATIVO"
    },
    {
        "id": "A6",
        "uidLivro": "L6",
        "tipoNegociacao": "TROCA",
        "categorias": ["Literatura Brasileira", "Romance"],
        "status": "FINALIZADO"
    }
])

anuncios

,id,uidLivro,tipoNegociacao,categorias,status
0,A1,L1,AMBOS,"[Fantasia, Aventura]",FINALIZADO
1,A2,L2,VENDA,"[Tecnologia, Programação]",FINALIZADO
2,A3,L3,TROCA,[Romance],ATIVO
3,A4,L4,AMBOS,[Fantasia],FINALIZADO
4,A5,L5,VENDA,[História],ATIVO
5,A6,L6,TROCA,"[Literatura Brasileira, Romance]",FINALIZADO


### Negociações simuladas

As negociações representam propostas realizadas entre os usuários a partir
dos anúncios disponíveis na plataforma.

O campo `anuncioId` permite relacionar cada negociação ao anúncio principal.
O campo `tipoNegociacao` identifica se a proposta foi realizada como TROCA
ou VENDA.

O status representa a situação atual da negociação, podendo assumir os
valores PENDENTE, EM_ANDAMENTO, RECUSADA, CANCELADA ou FINALIZADA.

A `dataNegociacao` registra o início da negociação. A `dataFinalizacao`,
por sua vez, é preenchida apenas quando uma negociação é concluída com
sucesso, permanecendo ausente nos demais status.

In [3]:
negociacoes = pd.DataFrame([
    {
        "id": "N1",
        "anuncioId": "A1",
        "anuncioOferecidoId": "A3",
        "dataNegociacao": "2026-08-03",
        "dataFinalizacao": "2026-08-07",
        "tipoNegociacao": "TROCA",
        "status": "FINALIZADA"
    },
    {
        "id": "N2",
        "anuncioId": "A2",
        "anuncioOferecidoId": None,
        "dataNegociacao": "2026-08-10",
        "dataFinalizacao": "2026-08-12",
        "tipoNegociacao": "VENDA",
        "status": "FINALIZADA"
    },
    {
        "id": "N3",
        "anuncioId": "A3",
        "anuncioOferecidoId": None,
        "dataNegociacao": "2026-08-15",
        "dataFinalizacao": None,
        "tipoNegociacao": "TROCA",
        "status": "RECUSADA"
    },
    {
        "id": "N4",
        "anuncioId": "A4",
        "anuncioOferecidoId": "A6",
        "dataNegociacao": "2026-09-02",
        "dataFinalizacao": "2026-09-09",
        "tipoNegociacao": "TROCA",
        "status": "FINALIZADA"
    },
    {
        "id": "N5",
        "anuncioId": "A5",
        "anuncioOferecidoId": None,
        "dataNegociacao": "2026-09-05",
        "dataFinalizacao": None,
        "tipoNegociacao": "VENDA",
        "status": "CANCELADA"
    },
    {
        "id": "N6",
        "anuncioId": "A5",
        "anuncioOferecidoId": None,
        "dataNegociacao": "2026-09-14",
        "dataFinalizacao": None,
        "tipoNegociacao": "VENDA",
        "status": "RECUSADA"
    },
    {
        "id": "N7",
        "anuncioId": "A6",
        "anuncioOferecidoId": None,
        "dataNegociacao": "2026-10-01",
        "dataFinalizacao": "2026-10-05",
        "tipoNegociacao": "TROCA",
        "status": "FINALIZADA"
    },
    {
        "id": "N8",
        "anuncioId": "A2",
        "anuncioOferecidoId": None,
        "dataNegociacao": "2026-10-08",
        "dataFinalizacao": None,
        "tipoNegociacao": "VENDA",
        "status": "PENDENTE"
    },
    {
        "id": "N9",
        "anuncioId": "A3",
        "anuncioOferecidoId": None,
        "dataNegociacao": "2026-10-11",
        "dataFinalizacao": None,
        "tipoNegociacao": "TROCA",
        "status": "EM_ANDAMENTO"
    },
    {
        "id": "N10",
        "anuncioId": "A5",
        "anuncioOferecidoId": None,
        "dataNegociacao": "2026-10-15",
        "dataFinalizacao": None,
        "tipoNegociacao": "VENDA",
        "status": "CANCELADA"
    }
])

negociacoes

,id,anuncioId,anuncioOferecidoId,dataNegociacao,dataFinalizacao,tipoNegociacao,status
0,N1,A1,A3,2026-08-03,2026-08-07,TROCA,FINALIZADA
1,N2,A2,NaN,2026-08-10,2026-08-12,VENDA,FINALIZADA
2,N3,A3,NaN,2026-08-15,NaN,TROCA,RECUSADA
3,N4,A4,A6,2026-09-02,2026-09-09,TROCA,FINALIZADA
4,N5,A5,NaN,2026-09-05,NaN,VENDA,CANCELADA
5,N6,A5,NaN,2026-09-14,NaN,VENDA,RECUSADA
6,N7,A6,NaN,2026-10-01,2026-10-05,TROCA,FINALIZADA
7,N8,A2,NaN,2026-10-08,NaN,VENDA,PENDENTE
8,N9,A3,NaN,2026-10-11,NaN,TROCA,EM_ANDAMENTO
9,N10,A5,NaN,2026-10-15,NaN,VENDA,CANCELADA


## 3. Preparação e tratamento dos dados

Antes de realizar as análises, é necessário verificar se os dados estão
organizados em um formato adequado para processamento.

Nesta etapa são analisados os tipos das colunas, a presença de valores
ausentes e a representação das datas.

Os valores ausentes em `dataFinalizacao` e `anuncioOferecidoId` não
representam necessariamente problemas de qualidade dos dados. Esses campos
podem permanecer vazios de acordo com as regras de negócio do Trocabook.

As datas, inicialmente representadas como texto nos dados simulados, serão
convertidas para o tipo `datetime` do Pandas para permitir análises
temporais nas próximas etapas.

### 3.1 Inspeção inicial dos dados

Inicialmente, é utilizada a função `info()` para verificar a quantidade de
registros, as colunas disponíveis, os tipos dos dados e a presença de
valores não nulos em cada DataFrame.

Essa inspeção permite identificar possíveis ajustes necessários antes das
análises.

In [4]:
anuncios.info()

<class 'pandas.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   id              6 non-null      str   
 1   uidLivro        6 non-null      str   
 2   tipoNegociacao  6 non-null      str   
 3   categorias      6 non-null      object
 4   status          6 non-null      str   
dtypes: object(1), str(4)
memory usage: 372.0+ bytes


In [5]:
negociacoes.info()

<class 'pandas.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 7 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   id                  10 non-null     str  
 1   anuncioId           10 non-null     str  
 2   anuncioOferecidoId  2 non-null      str  
 3   dataNegociacao      10 non-null     str  
 4   dataFinalizacao     4 non-null      str  
 5   tipoNegociacao      10 non-null     str  
 6   status              10 non-null     str  
dtypes: str(7)
memory usage: 692.0 bytes


### 3.2 Verificação de valores ausentes

A presença de valores ausentes é verificada para identificar campos que
podem exigir tratamento.

No conjunto de negociações, alguns valores ausentes são esperados pelas
regras de negócio da aplicação.

O campo `dataFinalizacao` permanece vazio enquanto uma negociação não foi
finalizada com sucesso.

O campo `anuncioOferecidoId` também pode permanecer vazio em negociações
de VENDA e em determinadas propostas de TROCA nas quais um livro é
oferecido sem possuir um anúncio associado.

In [6]:
anuncios.isnull().sum()

id                0
uidLivro          0
tipoNegociacao    0
categorias        0
status            0
dtype: int64

In [7]:
negociacoes.isnull().sum()

id                    0
anuncioId             0
anuncioOferecidoId    8
dataNegociacao        0
dataFinalizacao       6
tipoNegociacao        0
status                0
dtype: int64

### 3.3 Conversão das datas

As colunas `dataNegociacao` e `dataFinalizacao` foram inicialmente criadas
como texto.

Para permitir operações temporais, como agrupamento por período e cálculo
do tempo necessário para finalizar uma negociação, essas colunas são
convertidas para o tipo `datetime`.

Os valores ausentes de `dataFinalizacao` são mantidos pelo Pandas como
`NaT` (Not a Time), representação utilizada para datas ausentes.

In [8]:
negociacoes["dataNegociacao"] = pd.to_datetime(
    negociacoes["dataNegociacao"]
)

negociacoes["dataFinalizacao"] = pd.to_datetime(
    negociacoes["dataFinalizacao"]
)

negociacoes

,id,anuncioId,anuncioOferecidoId,dataNegociacao,dataFinalizacao,tipoNegociacao,status
0,N1,A1,A3,2026-08-03,2026-08-07,TROCA,FINALIZADA
1,N2,A2,NaN,2026-08-10,2026-08-12,VENDA,FINALIZADA
2,N3,A3,NaN,2026-08-15,NaT,TROCA,RECUSADA
3,N4,A4,A6,2026-09-02,2026-09-09,TROCA,FINALIZADA
4,N5,A5,NaN,2026-09-05,NaT,VENDA,CANCELADA
5,N6,A5,NaN,2026-09-14,NaT,VENDA,RECUSADA
6,N7,A6,NaN,2026-10-01,2026-10-05,TROCA,FINALIZADA
7,N8,A2,NaN,2026-10-08,NaT,VENDA,PENDENTE
8,N9,A3,NaN,2026-10-11,NaT,TROCA,EM_ANDAMENTO
9,N10,A5,NaN,2026-10-15,NaT,VENDA,CANCELADA


### 3.4 Verificação dos tipos após o tratamento

Após a conversão das datas, os tipos das colunas são verificados novamente.

Espera-se que `dataNegociacao` e `dataFinalizacao` estejam representadas
como `datetime64`, permitindo que sejam utilizadas em cálculos e
agrupamentos temporais.

In [9]:
negociacoes.dtypes

id                               str
anuncioId                        str
anuncioOferecidoId               str
dataNegociacao        datetime64[us]
dataFinalizacao       datetime64[us]
tipoNegociacao                   str
status                           str
dtype: object

## 4. Integração entre negociações e anúncios

As negociações armazenam o identificador do anúncio envolvido por meio do
campo `anuncioId`, porém não armazenam diretamente informações como as
categorias do livro.

Para permitir análises que relacionem o resultado das negociações às
características dos anúncios, é necessário combinar os dois conjuntos de
dados.

Essa integração é realizada utilizando a função `merge()` do Pandas,
relacionando o campo `anuncioId` das negociações ao campo `id` dos anúncios.

Será utilizado um `left join` para preservar todas as negociações, mesmo
caso algum anúncio não seja encontrado no conjunto de dados.

In [10]:
dados_negociacoes = negociacoes.merge(
    anuncios[
        [
            "id",
            "uidLivro",
            "categorias",
            "status"
        ]
    ],
    left_on="anuncioId",
    right_on="id",
    how="left",
    suffixes=("_negociacao", "_anuncio")
)

dados_negociacoes

,id_negociacao,anuncioId,anuncioOferecidoId,dataNegociacao,dataFinalizacao,tipoNegociacao,status_negociacao,id_anuncio,uidLivro,categorias,status_anuncio
0,N1,A1,A3,2026-08-03,2026-08-07,TROCA,FINALIZADA,A1,L1,"[Fantasia, Aventura]",FINALIZADO
1,N2,A2,NaN,2026-08-10,2026-08-12,VENDA,FINALIZADA,A2,L2,"[Tecnologia, Programação]",FINALIZADO
2,N3,A3,NaN,2026-08-15,NaT,TROCA,RECUSADA,A3,L3,[Romance],ATIVO
3,N4,A4,A6,2026-09-02,2026-09-09,TROCA,FINALIZADA,A4,L4,[Fantasia],FINALIZADO
4,N5,A5,NaN,2026-09-05,NaT,VENDA,CANCELADA,A5,L5,[História],ATIVO
5,N6,A5,NaN,2026-09-14,NaT,VENDA,RECUSADA,A5,L5,[História],ATIVO
6,N7,A6,NaN,2026-10-01,2026-10-05,TROCA,FINALIZADA,A6,L6,"[Literatura Brasileira, Romance]",FINALIZADO
7,N8,A2,NaN,2026-10-08,NaT,VENDA,PENDENTE,A2,L2,"[Tecnologia, Programação]",FINALIZADO
8,N9,A3,NaN,2026-10-11,NaT,TROCA,EM_ANDAMENTO,A3,L3,[Romance],ATIVO
9,N10,A5,NaN,2026-10-15,NaT,VENDA,CANCELADA,A5,L5,[História],ATIVO


### 4.1 Verificação da integração

Após a integração, é importante verificar se todas as negociações
encontraram seus respectivos anúncios.

Como foi utilizado um `left join`, uma negociação sem anúncio correspondente
continuaria presente no DataFrame, porém os campos provenientes do anúncio
apareceriam como valores ausentes.

Essa verificação ajuda a identificar possíveis inconsistências entre os
dados das coleções.

In [11]:
negociacoes_sem_anuncio = dados_negociacoes[
    dados_negociacoes["id_anuncio"].isna()
]

negociacoes_sem_anuncio

,id_negociacao,anuncioId,anuncioOferecidoId,dataNegociacao,dataFinalizacao,tipoNegociacao,status_negociacao,id_anuncio,uidLivro,categorias,status_anuncio


In [12]:
print(
    "Negociações:",
    len(negociacoes)
)

print(
    "Negociações após o merge:",
    len(dados_negociacoes)
)

print(
    "Negociações sem anúncio correspondente:",
    len(negociacoes_sem_anuncio)
)

Negociações: 10
Negociações após o merge: 10
Negociações sem anúncio correspondente: 0


### 4.2 Resultado da integração

A verificação demonstrou que a quantidade de negociações foi preservada
após o `merge` e que todas as negociações possuem um anúncio principal
correspondente no conjunto de dados simulado.

Dessa forma, as informações provenientes dos anúncios podem ser utilizadas
nas próximas análises sem perda de registros durante a integração.

Neste momento, o campo `anuncioOferecidoId` não é utilizado no relacionamento,
pois a análise geral considera o anúncio principal da negociação. A relação
com o anúncio oferecido será utilizada posteriormente em uma análise
específica das trocas.

## 5. Análise geral das negociações

Após a preparação e integração dos dados, é possível iniciar a análise
descritiva das negociações.

Nesta etapa são observadas a distribuição das negociações por status e por
tipo de negociação.

Esses indicadores fornecem uma visão geral do comportamento das negociações
na plataforma e servem como base para análises mais específicas, como a
comparação entre TROCA e VENDA e a análise das categorias dos livros.

### 5.1 Quantidade total de negociações

A quantidade total de negociações representa o número de propostas
registradas no conjunto de dados analisado.

Esse indicador fornece uma visão geral do volume de negociações e poderá
ser utilizado posteriormente como uma das métricas apresentadas no
Dashboard administrativo.

In [13]:
total_negociacoes = len(dados_negociacoes)

print(
    "Total de negociações:",
    total_negociacoes
)

Total de negociações: 10


### 5.2 Distribuição das negociações por status

A primeira análise verifica a quantidade de negociações em cada status.

Essa distribuição permite identificar quantas negociações foram finalizadas
com sucesso e quantas permanecem pendentes, em andamento, recusadas ou
canceladas.

In [14]:
negociacoes_por_status = (
    dados_negociacoes["status_negociacao"]
    .value_counts()
    .reset_index()
)

negociacoes_por_status.columns = [
    "status",
    "quantidade"
]

negociacoes_por_status

,status,quantidade
0,FINALIZADA,4
1,RECUSADA,2
2,CANCELADA,2
3,PENDENTE,1
4,EM_ANDAMENTO,1


### 5.3 Distribuição das negociações por tipo

Além do status, é importante verificar como as negociações estão
distribuídas entre TROCA e VENDA.

Essa análise permite observar a quantidade de negociações iniciadas de cada
tipo e servirá como base para comparar posteriormente suas respectivas
taxas de conclusão, cancelamento e recusa.

In [15]:
negociacoes_por_tipo = (
    dados_negociacoes["tipoNegociacao"]
    .value_counts()
    .reset_index()
)

negociacoes_por_tipo.columns = [
    "tipoNegociacao",
    "quantidade"
]

negociacoes_por_tipo

,tipoNegociacao,quantidade
0,TROCA,5
1,VENDA,5


## 6. Análise de TROCA e VENDA

Após a análise geral das negociações, é possível comparar o comportamento
dos dois tipos de negociação disponíveis no Trocabook: TROCA e VENDA.

Nesta etapa, o tipo da negociação será relacionado ao seu status, permitindo
identificar diferenças na quantidade de negociações finalizadas, recusadas,
canceladas, pendentes ou em andamento.

A partir desses valores também serão calculadas taxas que permitem comparar
o desempenho de TROCA e VENDA independentemente da quantidade total de
negociações de cada tipo.

### 6.1 Distribuição dos status por tipo de negociação

Inicialmente, as negociações são agrupadas pelo tipo de negociação e pelo
status.

Esse agrupamento permite observar como as negociações de TROCA e VENDA
estão distribuídas entre os diferentes estados possíveis da negociação.

In [16]:
status_por_tipo = (
    dados_negociacoes
    .groupby(
        [
            "tipoNegociacao",
            "status_negociacao"
        ]
    )
    .size()
    .reset_index(name="quantidade")
)

status_por_tipo

,tipoNegociacao,status_negociacao,quantidade
0,TROCA,EM_ANDAMENTO,1
1,TROCA,FINALIZADA,3
2,TROCA,RECUSADA,1
3,VENDA,CANCELADA,2
4,VENDA,FINALIZADA,1
5,VENDA,PENDENTE,1
6,VENDA,RECUSADA,1


### 6.2 Taxas por tipo de negociação

A comparação apenas pela quantidade absoluta de negociações pode produzir
interpretações inadequadas quando TROCA e VENDA possuem volumes diferentes.

Por esse motivo, são calculadas taxas considerando a quantidade total de
negociações de cada tipo.

A taxa de cada status é calculada dividindo sua quantidade pelo total de
negociações do respectivo tipo e multiplicando o resultado por 100.

Dessa forma, é possível observar proporcionalmente a distribuição dos
status de TROCA e VENDA e comparar indicadores como conclusão, cancelamento
e recusa entre os dois tipos de negociação.

In [17]:
totais_por_tipo = (
    dados_negociacoes
    .groupby("tipoNegociacao")
    .size()
)

totais_por_tipo

tipoNegociacao
TROCA    5
VENDA    5
dtype: int64

In [18]:
status_por_tipo["taxa"] = (
    status_por_tipo.apply(
        lambda linha:
            linha["quantidade"]
            / totais_por_tipo[linha["tipoNegociacao"]]
            * 100,
        axis=1
    )
)

status_por_tipo

,tipoNegociacao,status_negociacao,quantidade,taxa
0,TROCA,EM_ANDAMENTO,1,20.0
1,TROCA,FINALIZADA,3,60.0
2,TROCA,RECUSADA,1,20.0
3,VENDA,CANCELADA,2,40.0
4,VENDA,FINALIZADA,1,20.0
5,VENDA,PENDENTE,1,20.0
6,VENDA,RECUSADA,1,20.0


### 6.3 Comparação dos indicadores

Após calcular a distribuição percentual dos status, é possível comparar
diretamente os principais indicadores de TROCA e VENDA.

Nesta etapa são observadas as taxas de conclusão, cancelamento e recusa.
Também é calculada a diferença entre as taxas de conclusão dos dois tipos
de negociação.

A comparação por taxas permite identificar diferenças proporcionais entre
TROCA e VENDA, mesmo quando a quantidade de negociações de cada tipo for
diferente.

In [19]:
taxas_conclusao = status_por_tipo[
    status_por_tipo["status_negociacao"] == "FINALIZADA"
][
    [
        "tipoNegociacao",
        "taxa"
    ]
]

taxas_conclusao

,tipoNegociacao,taxa
1,TROCA,60.0
4,VENDA,20.0


### 6.4 Diferença entre as taxas de conclusão

Além das taxas individuais, é possível calcular a diferença entre a taxa
de conclusão das TROCAS e a taxa de conclusão das VENDAS.

Esse indicador permite quantificar a diferença observada no desempenho dos
dois tipos de negociação no conjunto de dados analisado.

A diferença é expressa em pontos percentuais, pois representa a subtração
entre duas porcentagens.

In [20]:
taxa_conclusao_troca = taxas_conclusao.loc[
    taxas_conclusao["tipoNegociacao"] == "TROCA",
    "taxa"
].iloc[0]

taxa_conclusao_venda = taxas_conclusao.loc[
    taxas_conclusao["tipoNegociacao"] == "VENDA",
    "taxa"
].iloc[0]

diferenca_taxa_conclusao = (
    taxa_conclusao_troca
    - taxa_conclusao_venda
)

print(
    "Taxa de conclusão das trocas:",
    f"{taxa_conclusao_troca:.2f}%"
)

print(
    "Taxa de conclusão das vendas:",
    f"{taxa_conclusao_venda:.2f}%"
)

print(
    "Diferença entre as taxas:",
    f"{diferenca_taxa_conclusao:.2f} pontos percentuais"
)

Taxa de conclusão das trocas: 60.00%
Taxa de conclusão das vendas: 20.00%
Diferença entre as taxas: 40.00 pontos percentuais


## 7. Análise por categoria

Após analisar o comportamento geral de TROCA e VENDA, as negociações podem
ser relacionadas às categorias dos livros anunciados.

Essa análise permite identificar quais categorias aparecem com maior
frequência nas negociações e, posteriormente, comparar seus resultados em
relação aos tipos e status das negociações.

Para manter a análise consistente entre TROCA e VENDA, são consideradas
apenas as categorias do anúncio principal associado ao campo `anuncioId`.

O anúncio oferecido em uma TROCA será analisado separadamente em uma etapa
específica sobre afinidade entre categorias.

### 7.1 Separação das categorias

Um anúncio pode possuir mais de uma categoria. Por esse motivo, o campo
`categorias` é armazenado como uma lista.

Para realizar agrupamentos e contagens por categoria, cada elemento dessa
lista precisa ser representado individualmente no DataFrame.

A função `explode()` do Pandas transforma cada elemento da lista em uma
linha separada, preservando as demais informações da negociação.

In [21]:
negociacoes_por_categoria = (
    dados_negociacoes
    .explode("categorias")
    .reset_index(drop=True)
)

negociacoes_por_categoria

,id_negociacao,anuncioId,anuncioOferecidoId,dataNegociacao,dataFinalizacao,tipoNegociacao,status_negociacao,id_anuncio,uidLivro,categorias,status_anuncio
0,N1,A1,A3,2026-08-03,2026-08-07,TROCA,FINALIZADA,A1,L1,Fantasia,FINALIZADO
1,N1,A1,A3,2026-08-03,2026-08-07,TROCA,FINALIZADA,A1,L1,Aventura,FINALIZADO
2,N2,A2,NaN,2026-08-10,2026-08-12,VENDA,FINALIZADA,A2,L2,Tecnologia,FINALIZADO
3,N2,A2,NaN,2026-08-10,2026-08-12,VENDA,FINALIZADA,A2,L2,Programação,FINALIZADO
4,N3,A3,NaN,2026-08-15,NaT,TROCA,RECUSADA,A3,L3,Romance,ATIVO
5,N4,A4,A6,2026-09-02,2026-09-09,TROCA,FINALIZADA,A4,L4,Fantasia,FINALIZADO
6,N5,A5,NaN,2026-09-05,NaT,VENDA,CANCELADA,A5,L5,História,ATIVO
7,N6,A5,NaN,2026-09-14,NaT,VENDA,RECUSADA,A5,L5,História,ATIVO
8,N7,A6,NaN,2026-10-01,2026-10-05,TROCA,FINALIZADA,A6,L6,Literatura Brasileira,FINALIZADO
9,N7,A6,NaN,2026-10-01,2026-10-05,TROCA,FINALIZADA,A6,L6,Romance,FINALIZADO


### 7.2 Quantidade de negociações por categoria

Após separar as categorias em registros individuais, é possível contabilizar
quantas negociações estão associadas a cada categoria.

Essa análise permite identificar as categorias com maior presença nas
negociações da plataforma.

Como um anúncio pode possuir mais de uma categoria, uma mesma negociação
pode contribuir para a contagem de mais de uma categoria.

In [22]:
quantidade_por_categoria = (
    negociacoes_por_categoria["categorias"]
    .value_counts()
    .reset_index()
)

quantidade_por_categoria.columns = [
    "categoria",
    "quantidade"
]

quantidade_por_categoria

,categoria,quantidade
0,Romance,3
1,História,3
2,Fantasia,2
3,Tecnologia,2
4,Programação,2
5,Aventura,1
6,Literatura Brasileira,1


### 7.3 Distribuição dos status por categoria

Após separar as categorias, é possível analisar como as negociações de cada
categoria estão distribuídas entre os diferentes status.

Essa análise permite observar, por exemplo, quantas negociações de uma
determinada categoria foram finalizadas, recusadas, canceladas, permanecem
pendentes ou estão em andamento.

Esses valores servirão como base para calcular posteriormente a taxa de
conclusão de cada categoria.

In [23]:
status_por_categoria = (
    negociacoes_por_categoria
    .groupby(
        [
            "categorias",
            "status_negociacao"
        ]
    )
    .size()
    .reset_index(name="quantidade")
)

status_por_categoria

,categorias,status_negociacao,quantidade
0,Aventura,FINALIZADA,1
1,Fantasia,FINALIZADA,2
2,História,CANCELADA,2
3,História,RECUSADA,1
4,Literatura Brasileira,FINALIZADA,1
5,Programação,FINALIZADA,1
6,Programação,PENDENTE,1
7,Romance,EM_ANDAMENTO,1
8,Romance,FINALIZADA,1
9,Romance,RECUSADA,1


### 7.4 Taxa de conclusão por categoria

A quantidade de negociações finalizadas por categoria não é suficiente para
comparar diretamente o desempenho entre categorias, pois algumas podem
possuir um volume maior de negociações do que outras.

Por esse motivo, é calculada a taxa de conclusão de cada categoria.

A taxa de conclusão representa a proporção de negociações com status
`FINALIZADA` em relação ao total de negociações associadas à respectiva
categoria.

Esse indicador permite comparar o comportamento histórico das categorias
independentemente da quantidade total de negociações de cada uma.

In [24]:
totais_por_categoria = (
    negociacoes_por_categoria
    .groupby("categorias")
    .size()
)

totais_por_categoria

categorias
Aventura                 1
Fantasia                 2
História                 3
Literatura Brasileira    1
Programação              2
Romance                  3
Tecnologia               2
dtype: int64

In [25]:
finalizadas_por_categoria = (
    status_por_categoria[
        status_por_categoria["status_negociacao"] == "FINALIZADA"
    ]
    .copy()
)

finalizadas_por_categoria

,categorias,status_negociacao,quantidade
0,Aventura,FINALIZADA,1
1,Fantasia,FINALIZADA,2
4,Literatura Brasileira,FINALIZADA,1
5,Programação,FINALIZADA,1
8,Romance,FINALIZADA,1
10,Tecnologia,FINALIZADA,1


In [26]:
finalizadas_por_categoria["taxaConclusao"] = (
    finalizadas_por_categoria.apply(
        lambda linha:
            linha["quantidade"]
            / totais_por_categoria[linha["categorias"]]
            * 100,
        axis=1
    )
)

finalizadas_por_categoria

,categorias,status_negociacao,quantidade,taxaConclusao
0,Aventura,FINALIZADA,1,100.000000
1,Fantasia,FINALIZADA,2,100.000000
4,Literatura Brasileira,FINALIZADA,1,100.000000
5,Programação,FINALIZADA,1,50.000000
8,Romance,FINALIZADA,1,33.333333
10,Tecnologia,FINALIZADA,1,50.000000


### 7.5 Consolidação dos indicadores por categoria

Para facilitar a comparação entre as categorias, os principais indicadores
são reunidos em uma única tabela.

A tabela apresenta a quantidade total de negociações associadas a cada
categoria, a quantidade de negociações finalizadas e a respectiva taxa de
conclusão.

Diferentemente da análise anterior, categorias que não possuem nenhuma
negociação finalizada também são preservadas no resultado, recebendo
quantidade de finalizações igual a zero.

Essa estrutura facilita a interpretação dos resultados e se aproxima do
formato que poderá ser utilizado posteriormente pelo microsserviço de
Analytics para fornecer indicadores ao Dashboard administrativo.

In [27]:
indicadores_categoria = (
    totais_por_categoria
    .reset_index(name="totalNegociacoes")
)

indicadores_categoria.columns = [
    "categoria",
    "totalNegociacoes"
]

indicadores_categoria

,categoria,totalNegociacoes
0,Aventura,1
1,Fantasia,2
2,História,3
3,Literatura Brasileira,1
4,Programação,2
5,Romance,3
6,Tecnologia,2


In [28]:
quantidade_finalizadas = (
    finalizadas_por_categoria[
        [
            "categorias",
            "quantidade"
        ]
    ]
    .rename(
        columns={
            "categorias": "categoria",
            "quantidade": "totalFinalizadas"
        }
    )
)

quantidade_finalizadas

,categoria,totalFinalizadas
0,Aventura,1
1,Fantasia,2
4,Literatura Brasileira,1
5,Programação,1
8,Romance,1
10,Tecnologia,1


In [29]:
indicadores_categoria = indicadores_categoria.merge(
    quantidade_finalizadas,
    on="categoria",
    how="left"
)

indicadores_categoria

,categoria,totalNegociacoes,totalFinalizadas
0,Aventura,1,1.0
1,Fantasia,2,2.0
2,História,3,NaN
3,Literatura Brasileira,1,1.0
4,Programação,2,1.0
5,Romance,3,1.0
6,Tecnologia,2,1.0


#### Tratamento das categorias sem finalizações

Após a integração dos indicadores, categorias sem negociações finalizadas
apresentam valor ausente na coluna `totalFinalizadas`.

Nesse contexto, o valor ausente não representa uma informação desconhecida.
Ele indica que nenhuma negociação daquela categoria foi finalizada.

Por esse motivo, esses valores são substituídos por zero.

In [30]:
indicadores_categoria["totalFinalizadas"] = (
    indicadores_categoria["totalFinalizadas"]
    .fillna(0)
    .astype(int)
)

indicadores_categoria

,categoria,totalNegociacoes,totalFinalizadas
0,Aventura,1,1
1,Fantasia,2,2
2,História,3,0
3,Literatura Brasileira,1,1
4,Programação,2,1
5,Romance,3,1
6,Tecnologia,2,1


In [31]:
indicadores_categoria["taxaConclusao"] = (
    indicadores_categoria["totalFinalizadas"]
    / indicadores_categoria["totalNegociacoes"]
    * 100
)

indicadores_categoria

,categoria,totalNegociacoes,totalFinalizadas,taxaConclusao
0,Aventura,1,1,100.000000
1,Fantasia,2,2,100.000000
2,História,3,0,0.000000
3,Literatura Brasileira,1,1,100.000000
4,Programação,2,1,50.000000
5,Romance,3,1,33.333333
6,Tecnologia,2,1,50.000000


### 7.6 Análise por categoria e tipo de negociação

A taxa de conclusão geral de uma categoria reúne todas as negociações
associadas a ela, independentemente de serem TROCA ou VENDA.

Entretanto, uma mesma categoria pode apresentar comportamentos diferentes
dependendo do tipo de negociação.

Por esse motivo, nesta etapa as negociações são agrupadas simultaneamente
por categoria e tipo de negociação, permitindo comparar o volume de
negociações e a taxa histórica de conclusão de cada combinação.

#### 7.6.1 Total de negociações por categoria e tipo

Inicialmente, é calculada a quantidade total de negociações para cada
combinação entre categoria e tipo de negociação.

Esse agrupamento permite identificar quantas negociações de TROCA e VENDA
estão associadas a cada categoria.

In [32]:
totais_categoria_tipo = (
    negociacoes_por_categoria
    .groupby(
        [
            "categorias",
            "tipoNegociacao"
        ]
    )
    .size()
    .reset_index(name="totalNegociacoes")
)

totais_categoria_tipo

,categorias,tipoNegociacao,totalNegociacoes
0,Aventura,TROCA,1
1,Fantasia,TROCA,2
2,História,VENDA,3
3,Literatura Brasileira,TROCA,1
4,Programação,VENDA,2
5,Romance,TROCA,3
6,Tecnologia,VENDA,2


#### 7.6.2 Negociações finalizadas por categoria e tipo

Em seguida, são consideradas apenas as negociações com status `FINALIZADA`.

Esses registros são agrupados novamente por categoria e tipo de negociação,
permitindo identificar quantas negociações foram concluídas com sucesso em
cada combinação.

In [33]:
finalizadas_categoria_tipo = (
    negociacoes_por_categoria[
        negociacoes_por_categoria["status_negociacao"] == "FINALIZADA"
    ]
    .groupby(
        [
            "categorias",
            "tipoNegociacao"
        ]
    )
    .size()
    .reset_index(name="totalFinalizadas")
)

finalizadas_categoria_tipo

,categorias,tipoNegociacao,totalFinalizadas
0,Aventura,TROCA,1
1,Fantasia,TROCA,2
2,Literatura Brasileira,TROCA,1
3,Programação,VENDA,1
4,Romance,TROCA,1
5,Tecnologia,VENDA,1


#### 7.6.3 Consolidação dos indicadores

Os totais de negociações e de finalizações são combinados para permitir o
cálculo da taxa de conclusão de cada categoria em cada tipo de negociação.

É utilizado um `left join` para preservar todas as combinações de categoria
e tipo que possuem negociações, mesmo quando nenhuma delas foi finalizada.

Nesses casos, a quantidade de finalizações é substituída por zero.

In [34]:
indicadores_categoria_tipo = totais_categoria_tipo.merge(
    finalizadas_categoria_tipo,
    on=[
        "categorias",
        "tipoNegociacao"
    ],
    how="left"
)

indicadores_categoria_tipo

,categorias,tipoNegociacao,totalNegociacoes,totalFinalizadas
0,Aventura,TROCA,1,1.0
1,Fantasia,TROCA,2,2.0
2,História,VENDA,3,NaN
3,Literatura Brasileira,TROCA,1,1.0
4,Programação,VENDA,2,1.0
5,Romance,TROCA,3,1.0
6,Tecnologia,VENDA,2,1.0


In [35]:
indicadores_categoria_tipo["totalFinalizadas"] = (
    indicadores_categoria_tipo["totalFinalizadas"]
    .fillna(0)
    .astype(int)
)

indicadores_categoria_tipo

,categorias,tipoNegociacao,totalNegociacoes,totalFinalizadas
0,Aventura,TROCA,1,1
1,Fantasia,TROCA,2,2
2,História,VENDA,3,0
3,Literatura Brasileira,TROCA,1,1
4,Programação,VENDA,2,1
5,Romance,TROCA,3,1
6,Tecnologia,VENDA,2,1


#### 7.6.4 Taxa de conclusão por categoria e tipo

Com os totais de negociações e finalizações disponíveis na mesma tabela,
é possível calcular diretamente a taxa histórica de conclusão de cada
combinação entre categoria e tipo de negociação.

A taxa é obtida dividindo a quantidade de negociações finalizadas pela
quantidade total de negociações da respectiva combinação e multiplicando
o resultado por 100.

In [36]:
indicadores_categoria_tipo["taxaConclusao"] = (
    indicadores_categoria_tipo["totalFinalizadas"]
    / indicadores_categoria_tipo["totalNegociacoes"]
    * 100
)

indicadores_categoria_tipo

,categorias,tipoNegociacao,totalNegociacoes,totalFinalizadas,taxaConclusao
0,Aventura,TROCA,1,1,100.000000
1,Fantasia,TROCA,2,2,100.000000
2,História,VENDA,3,0,0.000000
3,Literatura Brasileira,TROCA,1,1,100.000000
4,Programação,VENDA,2,1,50.000000
5,Romance,TROCA,3,1,33.333333
6,Tecnologia,VENDA,2,1,50.000000


## 8. Análise de afinidade entre livros em trocas

Nas análises anteriores foram consideradas apenas as categorias do anúncio
principal da negociação, permitindo comparar TROCA e VENDA de maneira
consistente.

Entretanto, uma negociação de TROCA pode envolver dois livros: o livro do
anúncio principal e o livro oferecido pelo usuário interessado.

Nesta etapa, as categorias desses dois livros são comparadas para identificar
possíveis padrões de afinidade entre os livros envolvidos nas trocas.

A análise busca verificar se trocas entre livros de categorias iguais,
semelhantes ou diferentes apresentam comportamentos distintos em relação
à finalização das negociações.

### 8.1 Seleção das negociações de TROCA

Como a análise de afinidade depende da comparação entre dois livros, são
selecionadas apenas as negociações do tipo `TROCA`.

As negociações de VENDA não participam desta etapa, pois não envolvem um
livro oferecido como parte da negociação.

In [37]:
trocas = dados_negociacoes[
    dados_negociacoes["tipoNegociacao"] == "TROCA"
].copy()

trocas

,id_negociacao,anuncioId,anuncioOferecidoId,dataNegociacao,dataFinalizacao,tipoNegociacao,status_negociacao,id_anuncio,uidLivro,categorias,status_anuncio
0,N1,A1,A3,2026-08-03,2026-08-07,TROCA,FINALIZADA,A1,L1,"[Fantasia, Aventura]",FINALIZADO
2,N3,A3,NaN,2026-08-15,NaT,TROCA,RECUSADA,A3,L3,[Romance],ATIVO
3,N4,A4,A6,2026-09-02,2026-09-09,TROCA,FINALIZADA,A4,L4,[Fantasia],FINALIZADO
6,N7,A6,NaN,2026-10-01,2026-10-05,TROCA,FINALIZADA,A6,L6,"[Literatura Brasileira, Romance]",FINALIZADO
8,N9,A3,NaN,2026-10-11,NaT,TROCA,EM_ANDAMENTO,A3,L3,[Romance],ATIVO


### 8.2 Seleção das trocas com anúncio oferecido

O campo `anuncioOferecidoId` identifica o anúncio do livro oferecido em uma
negociação de TROCA.

Entretanto, esse campo pode permanecer ausente em determinadas propostas.
Essas negociações continuam sendo válidas para outras análises, mas não
podem ser utilizadas na comparação entre categorias quando não existe um
anúncio oferecido associado.

Por esse motivo, nesta etapa são selecionadas apenas as trocas que possuem
`anuncioOferecidoId`.

In [38]:
trocas_com_anuncio_oferecido = trocas[
    trocas["anuncioOferecidoId"].notna()
].copy()

trocas_com_anuncio_oferecido

,id_negociacao,anuncioId,anuncioOferecidoId,dataNegociacao,dataFinalizacao,tipoNegociacao,status_negociacao,id_anuncio,uidLivro,categorias,status_anuncio
0,N1,A1,A3,2026-08-03,2026-08-07,TROCA,FINALIZADA,A1,L1,"[Fantasia, Aventura]",FINALIZADO
3,N4,A4,A6,2026-09-02,2026-09-09,TROCA,FINALIZADA,A4,L4,[Fantasia],FINALIZADO


### 8.3 Integração com o anúncio oferecido

As categorias do anúncio principal já estão disponíveis após a integração
realizada anteriormente entre negociações e anúncios.

Para obter também as categorias do livro oferecido, é realizado um novo
`merge` entre `anuncioOferecidoId` e o campo `id` dos anúncios.

Dessa forma, cada troca analisável passa a possuir as categorias do anúncio
principal e as categorias do anúncio oferecido, permitindo a comparação
entre os dois livros.

In [39]:
trocas_com_categorias = trocas_com_anuncio_oferecido.merge(
    anuncios[
        [
            "id",
            "uidLivro",
            "categorias"
        ]
    ],
    left_on="anuncioOferecidoId",
    right_on="id",
    how="left",
    suffixes=("_principal", "_oferecido")
).drop(
    columns=[
        "id",
        "id_anuncio"
    ]
)

trocas_com_categorias

,id_negociacao,anuncioId,anuncioOferecidoId,dataNegociacao,dataFinalizacao,tipoNegociacao,status_negociacao,uidLivro_principal,categorias_principal,status_anuncio,uidLivro_oferecido,categorias_oferecido
0,N1,A1,A3,2026-08-03,2026-08-07,TROCA,FINALIZADA,L1,"[Fantasia, Aventura]",FINALIZADO,L3,[Romance]
1,N4,A4,A6,2026-09-02,2026-09-09,TROCA,FINALIZADA,L4,[Fantasia],FINALIZADO,L6,"[Literatura Brasileira, Romance]"


### 8.4 Classificação da afinidade entre os livros

Com as categorias dos dois anúncios disponíveis, é possível comparar o livro
principal da negociação com o livro oferecido.

A afinidade entre os livros é classificada em três níveis:

- **MESMA:** os dois livros possuem exatamente o mesmo conjunto de categorias;
- **SEMELHANTE:** os livros possuem pelo menos uma categoria em comum, mas os
  conjuntos de categorias não são iguais;
- **DIFERENTE:** os livros não possuem categorias em comum.

Essa classificação permite analisar posteriormente se a afinidade entre os
livros apresenta alguma relação com a conclusão das negociações de troca.

In [40]:
def classificar_afinidade(categorias_principal, categorias_oferecido):
    categorias_principal = set(categorias_principal)
    categorias_oferecido = set(categorias_oferecido)

    if categorias_principal == categorias_oferecido:
        return "MESMA"

    if categorias_principal & categorias_oferecido:
        return "SEMELHANTE"

    return "DIFERENTE"

### 8.5 Aplicação da classificação de afinidade

A função de classificação é aplicada a cada negociação de troca que possui
um anúncio oferecido associado.

O resultado é armazenado em uma nova coluna chamada `afinidade`, permitindo
utilizar essa informação nas análises seguintes.

In [41]:
trocas_com_categorias["afinidade"] = (
    trocas_com_categorias.apply(
        lambda linha: classificar_afinidade(
            linha["categorias_principal"],
            linha["categorias_oferecido"]
        ),
        axis=1
    )
)

trocas_com_categorias[
    [
        "id_negociacao",
        "categorias_principal",
        "categorias_oferecido",
        "afinidade"
    ]
]

,id_negociacao,categorias_principal,categorias_oferecido,afinidade
0,N1,"[Fantasia, Aventura]",[Romance],DIFERENTE
1,N4,[Fantasia],"[Literatura Brasileira, Romance]",DIFERENTE


### 8.6 Análise da afinidade por status

Após classificar a afinidade entre os livros, as negociações são agrupadas
pela afinidade e pelo status da negociação.

Essa análise permite observar como negociações envolvendo livros com
diferentes níveis de afinidade foram distribuídas entre os possíveis
resultados, como finalização, cancelamento ou recusa.

In [42]:
status_por_afinidade = (
    trocas_com_categorias
    .groupby(["afinidade", "status_negociacao"])
    .size()
    .reset_index(name="quantidade")
)

status_por_afinidade

,afinidade,status_negociacao,quantidade
0,DIFERENTE,FINALIZADA,2


### 8.7 Taxa de conclusão por afinidade

Para comparar os diferentes níveis de afinidade, é calculada a taxa de
conclusão das negociações pertencentes a cada classificação.

A taxa representa a proporção de negociações finalizadas em relação ao
total de negociações analisáveis daquela afinidade.

Os resultados devem ser interpretados juntamente com a quantidade de
negociações observadas, pois grupos com poucas ocorrências podem apresentar
taxas elevadas sem possuir evidências suficientes para conclusões mais fortes.

In [43]:
indicadores_afinidade = (
    trocas_com_categorias
    .groupby("afinidade")
    .agg(
        totalNegociacoes=("id_negociacao", "count"),
        totalFinalizadas=(
            "status_negociacao",
            lambda status: (status == "FINALIZADA").sum()
        )
    )
    .reset_index()
)

indicadores_afinidade

,afinidade,totalNegociacoes,totalFinalizadas
0,DIFERENTE,2,2


In [44]:
indicadores_afinidade["taxaConclusao"] = (
    indicadores_afinidade["totalFinalizadas"]
    / indicadores_afinidade["totalNegociacoes"]
    * 100
)

indicadores_afinidade

,afinidade,totalNegociacoes,totalFinalizadas,taxaConclusao
0,DIFERENTE,2,2,100.0


## 9. Análise temporal

A análise temporal permite observar como as negociações se distribuem ao longo
do tempo, possibilitando identificar variações no volume de negociações e no
comportamento de TROCAS e VENDAS em diferentes períodos.

Também é analisado o tempo necessário para a conclusão das negociações
finalizadas, utilizando a diferença entre `dataNegociacao` e
`dataFinalizacao`.

Esses indicadores poderão ser utilizados posteriormente na identificação de
tendências e na geração de insights sobre o comportamento da plataforma.

### 9.1 Negociações iniciadas por mês

Para analisar a evolução do volume de negociações, a data de início de cada
negociação é convertida para um período mensal.

Em seguida, as negociações são agrupadas por mês, permitindo observar a
quantidade de negociações iniciadas em cada período.

In [45]:
dados_negociacoes["mesNegociacao"] = (
    dados_negociacoes["dataNegociacao"]
    .dt.to_period("M")
)

dados_negociacoes[
    [
        "id_negociacao",
        "dataNegociacao",
        "mesNegociacao"
    ]
]

,id_negociacao,dataNegociacao,mesNegociacao
0,N1,2026-08-03,2026-08
1,N2,2026-08-10,2026-08
2,N3,2026-08-15,2026-08
3,N4,2026-09-02,2026-09
4,N5,2026-09-05,2026-09
5,N6,2026-09-14,2026-09
6,N7,2026-10-01,2026-10
7,N8,2026-10-08,2026-10
8,N9,2026-10-11,2026-10
9,N10,2026-10-15,2026-10


In [46]:
negociacoes_por_mes = (
    dados_negociacoes
    .groupby("mesNegociacao")
    .size()
    .reset_index(name="quantidade")
)

negociacoes_por_mes

,mesNegociacao,quantidade
0,2026-08,3
1,2026-09,3
2,2026-10,4


### 9.2 Negociações por mês e tipo

Após analisar o volume geral de negociações por mês, as negociações são
agrupadas também pelo tipo de negociação.

Essa separação permite observar a evolução mensal de TROCAS e VENDAS,
identificando possíveis diferenças no comportamento das duas modalidades
ao longo do período analisado.

In [47]:
negociacoes_mes_tipo = (
    dados_negociacoes
    .groupby(["mesNegociacao", "tipoNegociacao"])
    .size()
    .reset_index(name="quantidade")
)

negociacoes_mes_tipo

,mesNegociacao,tipoNegociacao,quantidade
0,2026-08,TROCA,2
1,2026-08,VENDA,1
2,2026-09,TROCA,1
3,2026-09,VENDA,2
4,2026-10,TROCA,2
5,2026-10,VENDA,2


### 9.3 Negociações finalizadas por mês

Além da quantidade de negociações iniciadas em cada período, também é
importante analisar quando as negociações foram efetivamente concluídas.

Para essa análise, são consideradas somente as negociações com status
`FINALIZADA`, utilizando o campo `dataFinalizacao` para identificar o mês
em que a conclusão ocorreu.

O mês de finalização pode ser diferente do mês de início da negociação,
permitindo distinguir o volume de novas negociações do volume de
negociações efetivamente concluídas em cada período.

In [48]:
negociacoes_finalizadas = dados_negociacoes[
    dados_negociacoes["status_negociacao"] == "FINALIZADA"
].copy()

negociacoes_finalizadas[
    [
        "id_negociacao",
        "dataNegociacao",
        "dataFinalizacao",
        "tipoNegociacao"
    ]
]

,id_negociacao,dataNegociacao,dataFinalizacao,tipoNegociacao
0,N1,2026-08-03,2026-08-07,TROCA
1,N2,2026-08-10,2026-08-12,VENDA
3,N4,2026-09-02,2026-09-09,TROCA
6,N7,2026-10-01,2026-10-05,TROCA


In [49]:
negociacoes_finalizadas["mesFinalizacao"] = (
    negociacoes_finalizadas["dataFinalizacao"]
    .dt.to_period("M")
)

negociacoes_finalizadas[
    [
        "id_negociacao",
        "dataNegociacao",
        "dataFinalizacao",
        "mesFinalizacao"
    ]
]

,id_negociacao,dataNegociacao,dataFinalizacao,mesFinalizacao
0,N1,2026-08-03,2026-08-07,2026-08
1,N2,2026-08-10,2026-08-12,2026-08
3,N4,2026-09-02,2026-09-09,2026-09
6,N7,2026-10-01,2026-10-05,2026-10


In [50]:
finalizacoes_por_mes = (
    negociacoes_finalizadas
    .groupby("mesFinalizacao")
    .size()
    .reset_index(name="quantidade")
)

finalizacoes_por_mes

,mesFinalizacao,quantidade
0,2026-08,2
1,2026-09,1
2,2026-10,1


### 9.4 Tempo até a finalização

Além da quantidade de negociações concluídas, também é importante analisar
quanto tempo uma negociação leva para ser finalizada.

Para isso, é calculada a diferença entre `dataFinalizacao` e
`dataNegociacao` para cada negociação com status `FINALIZADA`.

Esse indicador permite observar a duração das negociações concluídas e,
posteriormente, comparar o tempo de finalização entre TROCAS e VENDAS.

#### 9.4.1 Cálculo do tempo de finalização

Para cada negociação finalizada, é calculada a diferença entre `dataFinalizacao` e `dataNegociacao`.

O resultado representa o intervalo de tempo transcorrido entre o início da negociação e sua conclusão.

In [51]:
negociacoes_finalizadas["tempoFinalizacao"] = (
    negociacoes_finalizadas["dataFinalizacao"]
    - negociacoes_finalizadas["dataNegociacao"]
)

negociacoes_finalizadas[
    [
        "id_negociacao",
        "dataNegociacao",
        "dataFinalizacao",
        "tempoFinalizacao"
    ]
]

,id_negociacao,dataNegociacao,dataFinalizacao,tempoFinalizacao
0,N1,2026-08-03,2026-08-07,4 days
1,N2,2026-08-10,2026-08-12,2 days
3,N4,2026-09-02,2026-09-09,7 days
6,N7,2026-10-01,2026-10-05,4 days


#### 9.4.2 Conversão do tempo de finalização para dias

A diferença entre as datas produz um valor do tipo `Timedelta`, que representa
uma duração.

Para facilitar cálculos estatísticos e comparações, essa duração é convertida
para a quantidade de dias entre o início e a finalização da negociação.

In [52]:
negociacoes_finalizadas["diasAteFinalizacao"] = (
    negociacoes_finalizadas["tempoFinalizacao"]
    .dt.days
)

negociacoes_finalizadas[
    [
        "id_negociacao",
        "tipoNegociacao",
        "diasAteFinalizacao"
    ]
]

,id_negociacao,tipoNegociacao,diasAteFinalizacao
0,N1,TROCA,4
1,N2,VENDA,2
3,N4,TROCA,7
6,N7,TROCA,4


#### 9.4.3 Tempo médio de finalização por tipo de negociação

Após calcular a duração de cada negociação finalizada, os registros são
agrupados pelo tipo de negociação para calcular o tempo médio necessário
para sua conclusão.

Essa análise permite comparar o tempo de finalização observado entre
TROCAS e VENDAS.

In [53]:
tempo_medio_por_tipo = (
    negociacoes_finalizadas
    .groupby("tipoNegociacao")
    .agg(
        totalFinalizadas=("id_negociacao", "count"),
        tempoMedioDias=("diasAteFinalizacao", "mean")
    )
    .reset_index()
)

tempo_medio_por_tipo

,tipoNegociacao,totalFinalizadas,tempoMedioDias
0,TROCA,3,5.0
1,VENDA,1,2.0


### 9.5 Taxa de conclusão por mês de início

Além da quantidade de negociações iniciadas e finalizadas em cada mês, é possível analisar a proporção das negociações iniciadas em determinado período que posteriormente atingiram o status `FINALIZADA`.

Diferentemente da análise de finalizações ocorridas por mês, esta análise mantém cada negociação associada ao mês em que foi iniciada. Dessa forma, uma negociação iniciada em um mês e finalizada posteriormente continua pertencendo ao grupo correspondente ao seu período de início.

Essa abordagem permite comparar historicamente a taxa de conclusão das negociações iniciadas em diferentes períodos.

Os períodos mais recentes devem ser interpretados com cautela, pois algumas negociações podem permanecer com status `PENDENTE` ou `EM_ANDAMENTO` simplesmente por ainda não terem tido tempo suficiente para atingir um estado definitivo.

#### 9.5.1 Total de negociações iniciadas por mês

Inicialmente, as negociações são agrupadas de acordo com o mês em que foram iniciadas.

Esse agrupamento fornece a quantidade total de negociações pertencentes a cada período e servirá como base para o cálculo posterior da taxa de conclusão mensal.

In [54]:
indicadores_temporais = (
    dados_negociacoes
    .groupby("mesNegociacao")
    .size()
    .reset_index(name="totalNegociacoes")
)

indicadores_temporais

,mesNegociacao,totalNegociacoes
0,2026-08,3
1,2026-09,3
2,2026-10,4


#### 9.5.2 Negociações finalizadas por mês de início

Em seguida, são selecionadas as negociações que atingiram o status `FINALIZADA` e elas são agrupadas de acordo com o mês em que foram iniciadas.

Dessa forma, a negociação permanece associada à sua coorte de início, independentemente do mês em que sua finalização ocorreu.

In [55]:
finalizadas_por_mes_inicio = (
    dados_negociacoes[
        dados_negociacoes["status_negociacao"] == "FINALIZADA"
    ]
    .groupby("mesNegociacao")
    .size()
    .reset_index(name="totalFinalizadas")
)

finalizadas_por_mes_inicio

,mesNegociacao,totalFinalizadas
0,2026-08,2
1,2026-09,1
2,2026-10,1


#### 9.5.3 Consolidação dos indicadores temporais
Os totais de negociações e de finalizações são reunidos em uma única estrutura para cada mês de início.

A partir desses valores, é calculada a taxa histórica de conclusão de cada período.

In [56]:
indicadores_temporais = indicadores_temporais.merge(
    finalizadas_por_mes_inicio,
    on="mesNegociacao",
    how="left"
)

In [57]:
indicadores_temporais["totalFinalizadas"] = (
    indicadores_temporais["totalFinalizadas"]
    .fillna(0)
    .astype(int)
)

In [58]:
indicadores_temporais["taxaConclusao"] = (
    indicadores_temporais["totalFinalizadas"]
    / indicadores_temporais["totalNegociacoes"]
    * 100
)

indicadores_temporais

,mesNegociacao,totalNegociacoes,totalFinalizadas,taxaConclusao
0,2026-08,3,2,66.666667
1,2026-09,3,1,33.333333
2,2026-10,4,1,25.000000


### 9.6 Taxa de conclusão por mês e tipo de negociação

A análise temporal também pode ser segmentada pelo tipo de negociação, permitindo observar separadamente o comportamento histórico de TROCA e VENDA em cada período.

Para cada combinação entre mês de início e tipo de negociação, são calculados o total de negociações iniciadas, o total posteriormente finalizado e a respectiva taxa de conclusão.

Essa análise permite verificar se as diferenças observadas entre TROCA e VENDA também aparecem ao longo dos períodos analisados.

Assim como na análise anterior, períodos recentes podem conter negociações ainda não concluídas e devem ser interpretados com cautela.

#### 9.6.1 Total por mês e tipo
As negociações são agrupadas simultaneamente pelo mês de início e pelo tipo de negociação.

Esse agrupamento permite identificar separadamente a quantidade de TROCAS e VENDAS iniciadas em cada período.

In [59]:
totais_mes_tipo = (
    dados_negociacoes
    .groupby(["mesNegociacao", "tipoNegociacao"])
    .size()
    .reset_index(name="totalNegociacoes")
)

totais_mes_tipo

,mesNegociacao,tipoNegociacao,totalNegociacoes
0,2026-08,TROCA,2
1,2026-08,VENDA,1
2,2026-09,TROCA,1
3,2026-09,VENDA,2
4,2026-10,TROCA,2
5,2026-10,VENDA,2


#### 9.6.2 Finalizadas por mês e tipo
As negociações finalizadas são agrupadas pelo mês de início e pelo tipo de negociação.

O resultado permite identificar quantas TROCAS e VENDAS iniciadas em cada período posteriormente atingiram o status `FINALIZADA`.

In [60]:
finalizadas_mes_tipo = (
    dados_negociacoes[
        dados_negociacoes["status_negociacao"] == "FINALIZADA"
    ]
    .groupby(["mesNegociacao", "tipoNegociacao"])
    .size()
    .reset_index(name="totalFinalizadas")
)

finalizadas_mes_tipo

,mesNegociacao,tipoNegociacao,totalFinalizadas
0,2026-08,TROCA,1
1,2026-08,VENDA,1
2,2026-09,TROCA,1
3,2026-10,TROCA,1


#### 9.6.3 Consolidação dos indicadores por mês e tipo

Os totais de negociações e de finalizações são combinados para cada relação entre mês de início e tipo de negociação.

Com esses valores, é calculada a taxa histórica de conclusão de TROCA e VENDA em cada período analisado.

In [61]:
indicadores_mes_tipo = totais_mes_tipo.merge(
    finalizadas_mes_tipo,
    on=["mesNegociacao", "tipoNegociacao"],
    how="left"
)

In [62]:
indicadores_mes_tipo["totalFinalizadas"] = (
    indicadores_mes_tipo["totalFinalizadas"]
    .fillna(0)
    .astype(int)
)

In [63]:
indicadores_mes_tipo["taxaConclusao"] = (
    indicadores_mes_tipo["totalFinalizadas"]
    / indicadores_mes_tipo["totalNegociacoes"]
    * 100
)

indicadores_mes_tipo

,mesNegociacao,tipoNegociacao,totalNegociacoes,totalFinalizadas,taxaConclusao
0,2026-08,TROCA,2,1,50.0
1,2026-08,VENDA,1,1,100.0
2,2026-09,TROCA,1,1,100.0
3,2026-09,VENDA,2,0,0.0
4,2026-10,TROCA,2,1,50.0
5,2026-10,VENDA,2,0,0.0


### 9.7 Limitações da análise temporal

As taxas de conclusão por período devem ser interpretadas considerando o tempo de observação disponível para cada grupo.

Períodos mais antigos tiveram mais tempo para que suas negociações atingissem um estado definitivo, enquanto períodos recentes podem apresentar negociações ainda classificadas como `PENDENTE` ou `EM_ANDAMENTO`.

Por esse motivo, uma taxa menor em um período recente não significa necessariamente que as negociações iniciadas nesse período apresentam pior desempenho.

Diferentemente da análise por categoria, não foi aplicado neste experimento o ajuste Beta-Binomial às taxas temporais. Embora a técnica pudesse reduzir a influência de períodos com poucas observações, ela não resolveria a diferença no tempo de acompanhamento das negociações.

Em análises futuras com dados históricos reais, poderá ser utilizada uma janela fixa de observação, como a proporção de negociações finalizadas em até determinado número de dias após o início. Dessa forma, diferentes períodos poderão ser comparados utilizando condições temporais equivalentes.

### 10. Confiabilidade e ajuste dos indicadores

As taxas de conclusão calculadas anteriormente representam diretamente o
comportamento observado nos dados. Entretanto, taxas baseadas em poucas
negociações podem apresentar valores elevados ou reduzidos sem possuir
quantidade suficiente de observações para sustentar uma comparação mais
confiável.

Por esse motivo, nesta etapa as taxas de conclusão são analisadas em conjunto
com o volume de negociações e, posteriormente, é calculada uma taxa ajustada
que reduz a influência de amostras muito pequenas.

A taxa histórica original é preservada, permitindo comparar o valor observado
com o indicador ajustado.

### 10.1 Taxa geral de conclusão

A taxa geral de conclusão representa a proporção de negociações finalizadas
em relação ao total de negociações registradas no conjunto analisado.

A taxa geral de conclusão fornece uma referência descritiva do comportamento histórico do conjunto analisado e poderá ser comparada posteriormente
com os indicadores calculados por categoria.

Esse valor não é utilizado diretamente no cálculo das taxas ajustadas pelo modelo Beta-Binomial, cuja distribuição de referência será estimada a partir dos resultados observados entre as categorias.

In [64]:
total_negociacoes = len(dados_negociacoes)

total_finalizadas = (
    dados_negociacoes["status_negociacao"] == "FINALIZADA"
).sum()

taxa_geral_conclusao = (
    total_finalizadas
    / total_negociacoes
    * 100
)

print(f"Total de negociações: {total_negociacoes}")
print(f"Total de finalizadas: {total_finalizadas}")
print(f"Taxa geral de conclusão: {taxa_geral_conclusao:.2f}%")

Total de negociações: 10
Total de finalizadas: 4
Taxa geral de conclusão: 40.00%


### 10.2 Comparação entre taxa de conclusão e volume de negociações

A taxa de conclusão de uma categoria deve ser interpretada juntamente com
a quantidade de negociações utilizadas em seu cálculo.

Uma categoria com poucas negociações pode apresentar uma taxa de conclusão
muito alta ou muito baixa devido à pequena quantidade de observações. Por
outro lado, taxas calculadas a partir de um volume maior de negociações
possuem maior quantidade de evidências históricas.

Por esse motivo, antes da aplicação de qualquer ajuste, são comparadas a
taxa de conclusão observada e a quantidade de negociações de cada categoria.

In [65]:
indicadores_categoria[
    [
        "categoria",
        "totalNegociacoes",
        "totalFinalizadas",
        "taxaConclusao"
    ]
].sort_values(
    by="taxaConclusao",
    ascending=False
)

,categoria,totalNegociacoes,totalFinalizadas,taxaConclusao
0,Aventura,1,1,100.000000
1,Fantasia,2,2,100.000000
3,Literatura Brasileira,1,1,100.000000
6,Tecnologia,2,1,50.000000
4,Programação,2,1,50.000000
5,Romance,3,1,33.333333
2,História,3,0,0.000000


### 10.3 Suavização das taxas com modelo Beta-Binomial

As taxas de conclusão observadas podem apresentar valores extremos quando uma categoria possui poucas negociações. Por exemplo, uma categoria com apenas uma negociação finalizada apresenta taxa de conclusão de 100%, embora exista pouca evidência para afirmar que esse comportamento representa a categoria de forma consistente.

Para reduzir esse problema, será utilizada uma abordagem de suavização baseada no modelo Beta-Binomial.

Diferentemente de uma análise baseada somente nas porcentagens observadas, esse modelo considera tanto a quantidade de negociações quanto a quantidade de finalizações de cada categoria. Dessa forma, taxas extremas provenientes de pequenas amostras possuem maior incerteza, enquanto categorias com maior quantidade de observações fornecem evidências mais consistentes.

A distribuição histórica será estimada a partir dos próprios dados, permitindo posteriormente calcular uma taxa de conclusão ajustada para cada categoria.

#### 10.3.1 Preparação dos dados para o modelo

Para estimar a distribuição histórica, são utilizadas duas informações de cada categoria:

- `totalNegociacoes`: quantidade de negociações observadas na categoria;
- `totalFinalizadas`: quantidade dessas negociações que foram finalizadas.

A taxa de conclusão observada é mantida para comparação posterior com a taxa ajustada.

O modelo utilizará diretamente as quantidades de negociações e finalizações, evitando tratar taxas iguais obtidas a partir de tamanhos de amostra diferentes como evidências equivalentes.

In [66]:
dados_beta_binomial = indicadores_categoria[
    [
        "categoria",
        "totalNegociacoes",
        "totalFinalizadas",
        "taxaConclusao"
    ]
].copy()

dados_beta_binomial

,categoria,totalNegociacoes,totalFinalizadas,taxaConclusao
0,Aventura,1,1,100.000000
1,Fantasia,2,2,100.000000
2,História,3,0,0.000000
3,Literatura Brasileira,1,1,100.000000
4,Programação,2,1,50.000000
5,Romance,3,1,33.333333
6,Tecnologia,2,1,50.000000


#### 10.3.2 Estimativa da distribuição histórica

Para calcular as taxas ajustadas, primeiro é necessário estimar uma referência histórica para o comportamento das categorias.

O modelo Beta-Binomial utiliza dois parâmetros, tradicionalmente denominados **alpha** e **beta**. Neste experimento, esses parâmetros serão chamados de `forca_finalizacao` e `forca_nao_finalizacao`, respectivamente, para facilitar sua interpretação.

- `forca_finalizacao`: representa a força histórica associada às finalizações;
- `forca_nao_finalizacao`: representa a força histórica associada às não finalizações.

Esses valores não serão definidos manualmente. Eles serão estimados a partir das quantidades de negociações e finalizações observadas em todas as categorias.

Para isso, será utilizada a máxima verossimilhança, buscando os valores de `forca_finalizacao` e `forca_nao_finalizacao` que tornam o conjunto de resultados observado mais plausível segundo o modelo Beta-Binomial.

In [67]:
finalizacoes_observadas = dados_beta_binomial[
    "totalFinalizadas"
].to_numpy()

negociacoes_observadas = dados_beta_binomial[
    "totalNegociacoes"
].to_numpy()

nao_finalizacoes_observadas = (
    negociacoes_observadas
    - finalizacoes_observadas
)

print("Finalizações:", finalizacoes_observadas)
print("Não finalizações:", nao_finalizacoes_observadas)
print("Negociações:", negociacoes_observadas)

Finalizações: [1 2 0 1 1 1 1]
Não finalizações: [0 0 3 0 1 2 1]
Negociações: [1 2 3 1 2 3 2]


In [68]:
def calcular_log_verossimilhanca_negativa(parametros):
    forca_finalizacao = parametros[0]
    forca_nao_finalizacao = parametros[1]

    log_verossimilhancas = (
        betaln(
            finalizacoes_observadas + forca_finalizacao,
            nao_finalizacoes_observadas + forca_nao_finalizacao
        )
        - betaln(
            forca_finalizacao,
            forca_nao_finalizacao
        )
    )

    log_verossimilhanca_total = log_verossimilhancas.sum()

    return -log_verossimilhanca_total

In [69]:
parametros_teste = [4, 6]

resultado_teste = calcular_log_verossimilhanca_negativa(
    parametros_teste
)

print(
    f"Log-verossimilhança negativa: "
    f"{resultado_teste:.4f}"
)

Log-verossimilhança negativa: 10.0119


In [70]:
resultado_otimizacao = minimize(
    calcular_log_verossimilhanca_negativa,
    x0=[1.0, 1.0],
    bounds=[
        (0.0001, None),
        (0.0001, None)
    ],
    method="L-BFGS-B"
)

resultado_otimizacao

  message: CONVERGENCE: NORM OF PROJECTED GRADIENT <= PGTOL
  success: True
   status: 0
      fun: 9.652714195137442
        x: [ 4.405e+00  4.055e+00]
      nit: 14
      jac: [-7.105e-07  0.000e+00]
     nfev: 51
     njev: 17
 hess_inv: <2x2 LbfgsInvHessProduct with dtype=float64>

#### 10.3.3 Parâmetros da distribuição histórica

Após a otimização, os parâmetros encontrados pelo modelo representam a força da evidência histórica associada às finalizações e às não finalizações.

A soma desses parâmetros representa a força total da distribuição de referência. A relação entre a força de finalização e a força total permite calcular a taxa central dessa distribuição.

Essa taxa de referência não corresponde necessariamente à taxa geral simples de conclusão da plataforma, pois ela é estimada pelo modelo Beta-Binomial considerando conjuntamente os resultados e os diferentes tamanhos de amostra das categorias.

In [71]:
forca_finalizacao = resultado_otimizacao.x[0]
forca_nao_finalizacao = resultado_otimizacao.x[1]

forca_referencia = (
    forca_finalizacao
    + forca_nao_finalizacao
)

taxa_referencia = (
    forca_finalizacao
    / forca_referencia
    * 100
)

print(f"Força de finalização: {forca_finalizacao:.4f}")
print(f"Força de não finalização: {forca_nao_finalizacao:.4f}")
print(f"Força total da referência: {forca_referencia:.4f}")
print(f"Taxa de referência: {taxa_referencia:.2f}%")

Força de finalização: 4.4050
Força de não finalização: 4.0553
Força total da referência: 8.4603
Taxa de referência: 52.07%


#### 10.3.4 Cálculo da taxa de conclusão ajustada

Após estimar os parâmetros da distribuição histórica, eles podem ser combinados com os resultados observados de cada categoria.

A taxa ajustada é calculada considerando as finalizações e negociações reais da categoria juntamente com as forças de finalização e não finalização estimadas pelo modelo Beta-Binomial.

Dessa forma, categorias com poucas observações são aproximadas da taxa de referência, enquanto categorias com maior quantidade de negociações possuem maior influência de seus próprios resultados.

A taxa ajustada é calculada pela expressão:

Taxa Ajustada = (Finalizações + Força de Finalização) / (Negociações + Força de Finalização + Força de Não Finalização)

In [72]:
dados_beta_binomial["taxaConclusaoAjustada"] = (
    (
        dados_beta_binomial["totalFinalizadas"]
        + forca_finalizacao
    )
    /
    (
        dados_beta_binomial["totalNegociacoes"]
        + forca_referencia
    )
    * 100
)

dados_beta_binomial

,categoria,totalNegociacoes,totalFinalizadas,taxaConclusao,taxaConclusaoAjustada
0,Aventura,1,1,100.000000,57.133027
1,Fantasia,2,2,100.000000,61.231093
2,História,3,0,0.000000,38.436645
3,Literatura Brasileira,1,1,100.000000,57.133027
4,Programação,2,1,50.000000,51.671133
5,Romance,3,1,33.333333,47.162424
6,Tecnologia,2,1,50.000000,51.671133


#### 10.3.5 Comparação entre taxa bruta e taxa ajustada

Para avaliar o efeito da suavização, as taxas de conclusão observadas são comparadas com as taxas ajustadas pelo modelo Beta-Binomial.

A comparação permite identificar o impacto do tamanho da amostra sobre os indicadores. Categorias com poucas negociações e taxas extremas tendem a sofrer maior ajuste em direção à taxa de referência, enquanto categorias com maior quantidade de observações tendem a preservar progressivamente seus próprios resultados.

A taxa ajustada fornece, portanto, uma estimativa mais cautelosa do comportamento histórico das categorias, evitando que pequenas amostras sejam interpretadas com a mesma confiança de categorias que possuem maior quantidade de negociações.

In [73]:
comparacao_taxas = dados_beta_binomial[
    [
        "categoria",
        "totalNegociacoes",
        "totalFinalizadas",
        "taxaConclusao",
        "taxaConclusaoAjustada"
    ]
].copy()

comparacao_taxas["diferencaAjuste"] = (
    comparacao_taxas["taxaConclusaoAjustada"]
    - comparacao_taxas["taxaConclusao"]
)

comparacao_taxas = comparacao_taxas.sort_values(
    by="taxaConclusaoAjustada",
    ascending=False
)

comparacao_taxas

,categoria,totalNegociacoes,totalFinalizadas,taxaConclusao,taxaConclusaoAjustada,diferencaAjuste
1,Fantasia,2,2,100.000000,61.231093,-38.768907
0,Aventura,1,1,100.000000,57.133027,-42.866973
3,Literatura Brasileira,1,1,100.000000,57.133027,-42.866973
6,Tecnologia,2,1,50.000000,51.671133,1.671133
4,Programação,2,1,50.000000,51.671133,1.671133
5,Romance,3,1,33.333333,47.162424,13.829091
2,História,3,0,0.000000,38.436645,38.436645


#### 10.3.6 Conclusão da suavização dos indicadores

A aplicação do modelo Beta-Binomial permitiu considerar o tamanho da amostra na interpretação das taxas de conclusão por categoria.

A análise demonstrou que taxas brutas extremas, como 0% ou 100%, podem ser pouco representativas quando são obtidas a partir de poucas negociações. A suavização aproxima esses valores da distribuição de referência estimada a partir do histórico das categorias, reduzindo a influência de pequenas amostras.

Ao mesmo tempo, conforme a quantidade de negociações de uma categoria aumenta, seus próprios resultados passam a exercer maior influência sobre a taxa ajustada. Dessa forma, categorias com maior volume de dados tendem a depender menos da referência histórica.

Neste experimento, a distribuição de referência estimada apresentou taxa central de aproximadamente 52,07%. Esse valor não representa a taxa geral de conclusão da plataforma, mas uma referência estatística estimada pelo modelo Beta-Binomial a partir do comportamento observado entre as categorias.

As taxas ajustadas também não devem ser interpretadas como probabilidades de conclusão de negociações futuras. Elas representam estimativas suavizadas do comportamento histórico das categorias e serão utilizadas como indicadores mais cautelosos nas análises posteriores.

Como o experimento utiliza um conjunto reduzido de dados simulados, os valores encontrados possuem finalidade de validação da metodologia. Na aplicação com os dados reais do Trocabook, a confiabilidade dos indicadores dependerá da quantidade e da diversidade das negociações registradas.

Como um anúncio pode possuir mais de uma categoria, uma mesma negociação pode contribuir para os indicadores de diferentes categorias após a aplicação de `explode()`. Dessa forma, os grupos de categorias analisados não são completamente independentes entre si.

Essa característica deve ser considerada como uma limitação metodológica do experimento, especialmente durante a interpretação da distribuição Beta-Binomial estimada. Nesta etapa, a técnica é utilizada para validar a estratégia de suavização dos indicadores, enquanto análises futuras com os dados reais poderão avaliar abordagens estatísticas mais adequadas à presença de categorias sobrepostas.

## 11. Geração de insights

Após o cálculo dos indicadores, os resultados podem ser interpretados para gerar insights automaticamente.

Os insights desta etapa são produzidos por regras baseadas nos indicadores calculados anteriormente. Dessa forma, o texto apresentado depende dos resultados encontrados nos dados, evitando conclusões fixas ou afirmações que não sejam sustentadas pelas análises.

Nesta etapa serão considerados indicadores relacionados ao tipo de negociação, categorias, tempo de finalização e afinidade entre os livros envolvidos em trocas.

### 11.1 Insight sobre TROCA e VENDA

A comparação entre TROCA e VENDA utiliza as taxas históricas de conclusão calculadas anteriormente.

O objetivo é identificar se um dos tipos apresentou maior taxa de conclusão no conjunto analisado e medir a diferença observada entre eles.

O insight representa uma interpretação descritiva dos dados históricos e não uma previsão sobre negociações futuras.

In [74]:
print(f"Taxa de conclusão de TROCA: {taxa_conclusao_troca:.2f}%")
print(f"Taxa de conclusão de VENDA: {taxa_conclusao_venda:.2f}%")
print(
    f"Diferença entre as taxas: "
    f"{abs(diferenca_taxa_conclusao):.2f} pontos percentuais"
)

Taxa de conclusão de TROCA: 60.00%
Taxa de conclusão de VENDA: 20.00%
Diferença entre as taxas: 40.00 pontos percentuais


In [75]:
if diferenca_taxa_conclusao > 0:
    insight_tipo_negociacao = (
        f"As negociações do tipo TROCA apresentaram taxa de conclusão "
        f"de {taxa_conclusao_troca:.2f}%, enquanto as negociações do tipo "
        f"VENDA apresentaram {taxa_conclusao_venda:.2f}%. "
        f"A diferença observada foi de "
        f"{abs(diferenca_taxa_conclusao):.2f} pontos percentuais "
        f"a favor das TROCAS."
    )

elif diferenca_taxa_conclusao < 0:
    insight_tipo_negociacao = (
        f"As negociações do tipo VENDA apresentaram taxa de conclusão "
        f"de {taxa_conclusao_venda:.2f}%, enquanto as negociações do tipo "
        f"TROCA apresentaram {taxa_conclusao_troca:.2f}%. "
        f"A diferença observada foi de "
        f"{abs(diferenca_taxa_conclusao):.2f} pontos percentuais "
        f"a favor das VENDAS."
    )

else:
    insight_tipo_negociacao = (
        f"As negociações dos tipos TROCA e VENDA apresentaram a mesma "
        f"taxa de conclusão, de {taxa_conclusao_troca:.2f}%."
    )

print(insight_tipo_negociacao)

As negociações do tipo TROCA apresentaram taxa de conclusão de 60.00%, enquanto as negociações do tipo VENDA apresentaram 20.00%. A diferença observada foi de 40.00 pontos percentuais a favor das TROCAS.


### 11.2 Insight sobre categorias

A análise das categorias utiliza as taxas de conclusão ajustadas pelo modelo Beta-Binomial.

A utilização da taxa ajustada reduz a influência de valores extremos provenientes de categorias com poucas negociações, permitindo uma comparação mais cautelosa entre os resultados observados.

O objetivo desta etapa é identificar as categorias que apresentaram as maiores e menores taxas ajustadas de conclusão no conjunto analisado.

Os resultados representam o comportamento histórico observado e não devem ser interpretados como previsões de conclusão de negociações futuras.

#### 11.2.1 Identificação das maiores e menores taxas ajustadas

Inicialmente, são identificadas as categorias que apresentam os maiores e menores valores de taxa de conclusão ajustada.

Essa identificação permite verificar os extremos encontrados antes da geração automática do insight.

In [76]:
indice_maior_taxa = (
    comparacao_taxas["taxaConclusaoAjustada"].idxmax()
)

indice_menor_taxa = (
    comparacao_taxas["taxaConclusaoAjustada"].idxmin()
)

categoria_maior_taxa = comparacao_taxas.loc[
    indice_maior_taxa
]

categoria_menor_taxa = comparacao_taxas.loc[
    indice_menor_taxa
]

In [77]:
print(
    f"Maior taxa ajustada: "
    f"{categoria_maior_taxa['categoria']} - "
    f"{categoria_maior_taxa['taxaConclusaoAjustada']:.2f}%"
)

print(
    f"Menor taxa ajustada: "
    f"{categoria_menor_taxa['categoria']} - "
    f"{categoria_menor_taxa['taxaConclusaoAjustada']:.2f}%"
)

Maior taxa ajustada: Fantasia - 61.23%
Menor taxa ajustada: História - 38.44%


#### 11.2.2 Geração do insight sobre categorias

A partir das taxas ajustadas, é possível gerar automaticamente um insight destacando as categorias que apresentaram os maiores e menores indicadores de conclusão.

Como diferentes categorias podem apresentar a mesma taxa ajustada, a análise considera todas as categorias empatadas no maior ou no menor valor encontrado.

O insight mantém caráter descritivo, apresentando somente o comportamento observado no conjunto analisado.

In [78]:
maior_taxa_ajustada = (
    comparacao_taxas["taxaConclusaoAjustada"].max()
)

menor_taxa_ajustada = (
    comparacao_taxas["taxaConclusaoAjustada"].min()
)

In [79]:
categorias_maior_taxa = comparacao_taxas[
    comparacao_taxas["taxaConclusaoAjustada"]
    == maior_taxa_ajustada
]

categorias_menor_taxa = comparacao_taxas[
    comparacao_taxas["taxaConclusaoAjustada"]
    == menor_taxa_ajustada
]

In [80]:
nomes_maiores_taxas = ", ".join(
    categorias_maior_taxa["categoria"].tolist()
)

nomes_menores_taxas = ", ".join(
    categorias_menor_taxa["categoria"].tolist()
)

In [81]:
insight_categorias = (
    f"A categoria ou conjunto de categorias com maior taxa de conclusão "
    f"ajustada foi {nomes_maiores_taxas}, com "
    f"{maior_taxa_ajustada:.2f}%. "
    f"A menor taxa ajustada foi observada em "
    f"{nomes_menores_taxas}, com "
    f"{menor_taxa_ajustada:.2f}%."
)

print(insight_categorias)

A categoria ou conjunto de categorias com maior taxa de conclusão ajustada foi Fantasia, com 61.23%. A menor taxa ajustada foi observada em História, com 38.44%.


### 11.3 Insight sobre tempo de finalização

O tempo médio de finalização permite comparar quanto tempo as negociações concluídas levaram entre o início da negociação e sua finalização.

A análise considera somente negociações com status `FINALIZADA`, pois apenas elas possuem uma data de finalização válida.

O objetivo é identificar diferenças no tempo médio de conclusão entre TROCA e VENDA. Como a quantidade de negociações finalizadas pode variar entre os tipos, o número de observações também deve ser considerado durante a interpretação dos resultados.

Os valores encontrados representam somente o comportamento histórico das negociações finalizadas analisadas.

In [82]:
tempo_troca = tempo_medio_por_tipo[
    tempo_medio_por_tipo["tipoNegociacao"] == "TROCA"
].iloc[0]

tempo_venda = tempo_medio_por_tipo[
    tempo_medio_por_tipo["tipoNegociacao"] == "VENDA"
].iloc[0]

In [83]:
print(
    f"TROCA: {tempo_troca['tempoMedioDias']:.2f} dias "
    f"({int(tempo_troca['totalFinalizadas'])} finalizadas)"
)

print(
    f"VENDA: {tempo_venda['tempoMedioDias']:.2f} dias "
    f"({int(tempo_venda['totalFinalizadas'])} finalizadas)"
)

TROCA: 5.00 dias (3 finalizadas)
VENDA: 2.00 dias (1 finalizadas)


In [84]:
diferenca_tempo_medio = abs(
    tempo_troca["tempoMedioDias"]
    - tempo_venda["tempoMedioDias"]
)

In [85]:
if tempo_troca["tempoMedioDias"] < tempo_venda["tempoMedioDias"]:
    tipo_mais_rapido = "TROCA"
elif tempo_venda["tempoMedioDias"] < tempo_troca["tempoMedioDias"]:
    tipo_mais_rapido = "VENDA"
else:
    tipo_mais_rapido = None

In [86]:
total_trocas_finalizadas = int(
    tempo_troca["totalFinalizadas"]
)

total_vendas_finalizadas = int(
    tempo_venda["totalFinalizadas"]
)

texto_trocas = (
    f"{total_trocas_finalizadas} "
    f"{'TROCA finalizada' if total_trocas_finalizadas == 1 else 'TROCAS finalizadas'}"
)

texto_vendas = (
    f"{total_vendas_finalizadas} "
    f"{'VENDA finalizada' if total_vendas_finalizadas == 1 else 'VENDAS finalizadas'}"
)

In [87]:
if tipo_mais_rapido is None:
    insight_tempo_finalizacao = (
        f"As negociações finalizadas de TROCA e VENDA apresentaram "
        f"o mesmo tempo médio de finalização, de "
        f"{tempo_troca['tempoMedioDias']:.2f} dias. "
        f"A análise considerou "
        f"{texto_trocas} e "
        f"{texto_vendas}."
    )
else:
    insight_tempo_finalizacao = (
        f"As negociações finalizadas do tipo {tipo_mais_rapido} "
        f"apresentaram menor tempo médio de finalização. "
        f"As TROCAS apresentaram média de "
        f"{tempo_troca['tempoMedioDias']:.2f} dias, enquanto as "
        f"VENDAS apresentaram {tempo_venda['tempoMedioDias']:.2f} dias. "
        f"A diferença observada foi de "
        f"{diferenca_tempo_medio:.2f} dias. "
        f"A análise considerou "
        f"{texto_trocas} e "
        f"{texto_vendas}."
    )

print(insight_tempo_finalizacao)

As negociações finalizadas do tipo VENDA apresentaram menor tempo médio de finalização. As TROCAS apresentaram média de 5.00 dias, enquanto as VENDAS apresentaram 2.00 dias. A diferença observada foi de 3.00 dias. A análise considerou 3 TROCAS finalizadas e 1 VENDA finalizada.


### 11.4 Insight sobre evolução temporal

A análise temporal permite observar como as taxas históricas de conclusão variaram entre os períodos de início das negociações.

Para cada mês, são considerados o total de negociações iniciadas e a quantidade que posteriormente atingiu o status `FINALIZADA`.

Entretanto, os períodos analisados podem possuir tempos de acompanhamento diferentes. Negociações iniciadas em períodos mais recentes podem permanecer como `PENDENTE` ou `EM_ANDAMENTO` simplesmente por ainda não terem tido tempo suficiente para atingir um estado definitivo.

Por esse motivo, o insight temporal apresenta as diferenças observadas entre os períodos, mas não interpreta automaticamente aumentos ou reduções nas taxas como uma tendência de melhora ou piora da plataforma.

#### 11.4.1 Identificação dos períodos extremos

A partir dos indicadores temporais calculados anteriormente, são identificados os períodos que apresentaram as maiores e menores taxas históricas de conclusão.

A identificação considera também a possibilidade de diferentes períodos apresentarem a mesma taxa.

In [88]:
maior_taxa_temporal = (
    indicadores_temporais["taxaConclusao"].max()
)

menor_taxa_temporal = (
    indicadores_temporais["taxaConclusao"].min()
)

In [89]:
periodos_maior_taxa = indicadores_temporais[
    indicadores_temporais["taxaConclusao"]
    == maior_taxa_temporal
]

periodos_menor_taxa = indicadores_temporais[
    indicadores_temporais["taxaConclusao"]
    == menor_taxa_temporal
]

In [90]:
nomes_periodos_maior_taxa = ", ".join(
    periodos_maior_taxa["mesNegociacao"]
    .astype(str)
    .tolist()
)

nomes_periodos_menor_taxa = ", ".join(
    periodos_menor_taxa["mesNegociacao"]
    .astype(str)
    .tolist()
)

#### 11.4.2 Geração do insight temporal

Com os períodos identificados, é gerado automaticamente um insight que apresenta os maiores e menores valores observados.

A interpretação não classifica essas diferenças como tendência de melhora ou piora, pois os períodos podem possuir tempos de acompanhamento diferentes.

In [91]:
insight_evolucao_temporal = (
    f"O período com maior taxa histórica de conclusão entre as "
    f"negociações iniciadas foi {nomes_periodos_maior_taxa}, com "
    f"{maior_taxa_temporal:.2f}%. "
    f"A menor taxa observada ocorreu em "
    f"{nomes_periodos_menor_taxa}, com "
    f"{menor_taxa_temporal:.2f}%. "
    f"Essas diferenças não devem ser interpretadas automaticamente "
    f"como uma tendência de melhora ou piora, pois os períodos mais "
    f"recentes podem possuir negociações que ainda não tiveram tempo "
    f"suficiente para atingir um estado definitivo."
)

print(insight_evolucao_temporal)

O período com maior taxa histórica de conclusão entre as negociações iniciadas foi 2026-08, com 66.67%. A menor taxa observada ocorreu em 2026-10, com 25.00%. Essas diferenças não devem ser interpretadas automaticamente como uma tendência de melhora ou piora, pois os períodos mais recentes podem possuir negociações que ainda não tiveram tempo suficiente para atingir um estado definitivo.


### 11.5 Insight sobre afinidade entre livros nas trocas

A análise de afinidade busca verificar se a relação entre as categorias do livro principal e do livro oferecido apresenta diferenças nos resultados históricos das trocas.

As trocas são classificadas como `MESMA`, `SEMELHANTE` ou `DIFERENTE`, conforme a relação entre as categorias dos dois livros.

Além da taxa de conclusão, a quantidade de negociações observadas em cada classificação deve ser considerada. Taxas elevadas obtidas a partir de poucas observações não fornecem evidência suficiente para estabelecer uma relação consistente entre afinidade e conclusão.

O insight gerado nesta etapa possui caráter descritivo e não estabelece que determinada afinidade cause maior ou menor conclusão das negociações.

In [92]:
quantidade_afinidades_observadas = (
    indicadores_afinidade["afinidade"].nunique()
)

quantidade_trocas_com_afinidade = (
    indicadores_afinidade["totalNegociacoes"].sum()
)

print(
    f"Classificações de afinidade observadas: "
    f"{quantidade_afinidades_observadas}"
)

print(
    f"Trocas analisadas: "
    f"{quantidade_trocas_com_afinidade}"
)

Classificações de afinidade observadas: 1
Trocas analisadas: 2


In [93]:
if quantidade_afinidades_observadas < 2:
    afinidade_observada = indicadores_afinidade.iloc[0]

    insight_afinidade = (
        f"Foram analisadas "
        f"{int(quantidade_trocas_com_afinidade)} trocas com informação "
        f"de afinidade entre os livros. "
        f"Todas as observações pertencem à classificação "
        f"{afinidade_observada['afinidade']}, que apresentou taxa "
        f"histórica de conclusão de "
        f"{afinidade_observada['taxaConclusao']:.2f}%. "
        f"Como apenas uma classificação de afinidade foi observada, "
        f"não existem dados suficientes para comparar a relação entre "
        f"a afinidade e a conclusão das trocas."
    )

else:
    maior_taxa_afinidade = (
        indicadores_afinidade["taxaConclusao"].max()
    )

    afinidades_maior_taxa = indicadores_afinidade[
        indicadores_afinidade["taxaConclusao"]
        == maior_taxa_afinidade
    ]

    nomes_afinidades = ", ".join(
        afinidades_maior_taxa["afinidade"].tolist()
    )

    insight_afinidade = (
        f"A classificação de afinidade com maior taxa histórica "
        f"de conclusão foi {nomes_afinidades}, com "
        f"{maior_taxa_afinidade:.2f}%. "
        f"A comparação considera somente as classificações observadas "
        f"no conjunto analisado."
    )

print(insight_afinidade)

Foram analisadas 2 trocas com informação de afinidade entre os livros. Todas as observações pertencem à classificação DIFERENTE, que apresentou taxa histórica de conclusão de 100.00%. Como apenas uma classificação de afinidade foi observada, não existem dados suficientes para comparar a relação entre a afinidade e a conclusão das trocas.


### 11.6 Consolidação dos insights

Após a geração individual dos insights, os resultados são consolidados em uma única estrutura.

Essa organização facilita o consumo das análises por outras partes do sistema e aproxima o experimento da estrutura que poderá ser utilizada futuramente pelo serviço de Analytics.

Cada insight possui um tipo que identifica a análise realizada e uma descrição contendo a interpretação produzida a partir dos indicadores históricos.

In [94]:
insights = [
    {
        "tipo": "TIPO_NEGOCIACAO",
        "descricao": insight_tipo_negociacao
    },
    {
        "tipo": "CATEGORIA",
        "descricao": insight_categorias
    },
    {
        "tipo": "TEMPO_FINALIZACAO",
        "descricao": insight_tempo_finalizacao
    },
    {
        "tipo": "EVOLUCAO_TEMPORAL",
        "descricao": insight_evolucao_temporal
    },
    {
        "tipo": "AFINIDADE",
        "descricao": insight_afinidade
    }
]

insights

[{'tipo': 'TIPO_NEGOCIACAO',
  'descricao': 'As negociações do tipo TROCA apresentaram taxa de conclusão de 60.00%, enquanto as negociações do tipo VENDA apresentaram 20.00%. A diferença observada foi de 40.00 pontos percentuais a favor das TROCAS.'},
 {'tipo': 'CATEGORIA',
  'descricao': 'A categoria ou conjunto de categorias com maior taxa de conclusão ajustada foi Fantasia, com 61.23%. A menor taxa ajustada foi observada em História, com 38.44%.'},
 {'tipo': 'TEMPO_FINALIZACAO',
  'descricao': 'As negociações finalizadas do tipo VENDA apresentaram menor tempo médio de finalização. As TROCAS apresentaram média de 5.00 dias, enquanto as VENDAS apresentaram 2.00 dias. A diferença observada foi de 3.00 dias. A análise considerou 3 TROCAS finalizadas e 1 VENDA finalizada.'},
 {'tipo': 'EVOLUCAO_TEMPORAL',
  'descricao': 'O período com maior taxa histórica de conclusão entre as negociações iniciadas foi 2026-08, com 66.67%. A menor taxa observada ocorreu em 2026-10, com 25.00%. Essas dife

In [95]:
for insight in insights:
    print(f"[{insight['tipo']}]")
    print(insight["descricao"])
    print()

[TIPO_NEGOCIACAO]
As negociações do tipo TROCA apresentaram taxa de conclusão de 60.00%, enquanto as negociações do tipo VENDA apresentaram 20.00%. A diferença observada foi de 40.00 pontos percentuais a favor das TROCAS.

[CATEGORIA]
A categoria ou conjunto de categorias com maior taxa de conclusão ajustada foi Fantasia, com 61.23%. A menor taxa ajustada foi observada em História, com 38.44%.

[TEMPO_FINALIZACAO]
As negociações finalizadas do tipo VENDA apresentaram menor tempo médio de finalização. As TROCAS apresentaram média de 5.00 dias, enquanto as VENDAS apresentaram 2.00 dias. A diferença observada foi de 3.00 dias. A análise considerou 3 TROCAS finalizadas e 1 VENDA finalizada.

[EVOLUCAO_TEMPORAL]
O período com maior taxa histórica de conclusão entre as negociações iniciadas foi 2026-08, com 66.67%. A menor taxa observada ocorreu em 2026-10, com 25.00%. Essas diferenças não devem ser interpretadas automaticamente como uma tendência de melhora ou piora, pois os períodos mais r

### 11.7 Conclusão da geração de insights

A geração automática de insights demonstrou que os indicadores calculados ao longo do experimento podem ser transformados em interpretações textuais sem depender de conclusões previamente definidas.

As regras implementadas analisam os resultados encontrados nos dados antes de produzir cada descrição. Dessa forma, alterações futuras nos dados de negociações também poderão modificar automaticamente os insights apresentados.

Foram gerados insights relacionados à comparação entre TROCA e VENDA, às taxas ajustadas por categoria, ao tempo médio de finalização, à evolução temporal das taxas de conclusão e à afinidade entre os livros envolvidos nas trocas.

A análise também demonstrou a importância de considerar a quantidade de observações antes de produzir conclusões. Quando os dados disponíveis não permitem uma comparação adequada, como ocorreu na análise de afinidade deste experimento, o sistema informa a limitação em vez de estabelecer uma relação que não seja sustentada pelos dados.

Os insights produzidos possuem caráter descritivo e diagnóstico. Eles representam interpretações dos dados históricos analisados e não constituem previsões sobre o resultado de negociações futuras.

A estrutura consolidada dos insights também poderá ser reutilizada posteriormente pelo serviço de Analytics, permitindo disponibilizar essas informações por meio da API para apresentação no Dashboard administrativo do Trocabook.

## 12. Conclusão do experimento

O experimento permitiu validar uma abordagem para análise descritiva e diagnóstica das negociações realizadas no Trocabook.

A partir da integração entre os dados de anúncios e negociações, foi possível calcular indicadores relacionados à quantidade e ao status das negociações, comparar o comportamento de TROCA e VENDA, analisar os resultados por categoria, avaliar a afinidade entre os livros envolvidos em trocas e observar a evolução temporal e o tempo de finalização das negociações.

Durante a análise por categoria, foi identificado que taxas de conclusão calculadas a partir de pequenas quantidades de negociações podem produzir valores extremos e pouco representativos. Para reduzir esse efeito, foi aplicado um modelo Beta-Binomial, cujos parâmetros foram estimados por máxima verossimilhança. A utilização da taxa ajustada permitiu considerar conjuntamente a taxa de conclusão observada e a quantidade de evidências disponíveis para cada categoria.

Os indicadores calculados também foram utilizados para validar uma estratégia de geração automática de insights. As regras implementadas produzem interpretações de acordo com os resultados encontrados e consideram limitações dos dados antes de estabelecer comparações. Quando as informações disponíveis não são suficientes para sustentar uma conclusão, o sistema pode apresentar essa limitação em vez de produzir uma afirmação sem evidências adequadas.

Os resultados deste notebook possuem caráter descritivo e diagnóstico. As taxas ajustadas e os insights gerados representam interpretações do comportamento histórico analisado e não devem ser considerados previsões sobre negociações futuras.

Como o experimento foi desenvolvido com um conjunto reduzido de dados simulados, os valores obtidos não representam o comportamento real dos usuários do Trocabook. O objetivo desta etapa foi validar a metodologia, os indicadores e as regras de análise que poderão ser posteriormente aplicados aos dados reais armazenados no Firebase Cloud Firestore.

A lógica validada neste experimento poderá ser incorporada ao serviço de Analytics do Trocabook, permitindo que os indicadores e insights sejam disponibilizados por meio de uma API e posteriormente apresentados no Dashboard administrativo da plataforma.

Experimentos posteriores poderão utilizar os dados históricos reais para avaliar modelos preditivos, incluindo a previsão da evolução da quantidade de usuários e a estimativa da probabilidade de finalização de novas negociações.